In [11]:
#by LLZ
#on 14/10/2021

# updata Jingyi
# ON 24/ June/ 2025
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import tabula
from time import sleep
import os
from tabula.io import read_pdf

    

In [12]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'BO ASFI' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.2.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running BO ASFI Web Scraping Tool v.2.0


In [13]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [14]:
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')
regdict = {'BO ASFI 1': 'https://www.asfi.gob.bo/index.php/int-fin-entidades-supervisadas/int-fin-entidades-de-intermediacion-con-licencia-de-funcionamiento.html'}

Typology={ regulatorName+' 1': 'Entidades Supervisadas Con Licencia De Funcionamiento'}

In [15]:
def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

In [ ]:
import pdfplumber

for reg in regdict:
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(4)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    element1 = driver.find_element(By.PARTIAL_LINK_TEXT, 'de funcionamiento')
    driver.execute_script("arguments[0].click();", element1)
    sleep(10)

    # Get the first PDF file in tempfolder (full path)
    pdf_files = [f for f in os.listdir(tempfolder) if f.lower().endswith('.pdf')]
    if not pdf_files:
        raise FileNotFoundError("No PDF file found in tempfolder.")
    pdf_path = os.path.join(tempfolder, pdf_files[0])
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_tables = page.extract_tables()
            tables.extend(page_tables)
            
    for j in range(len(tables)):
        for i in tables[j][:]:
            try:
                if i[0].isdigit():
                    print(i[0])
                    name = i[1]
                    #print(name)
                    city = i[2]
                    #print(city)
                    sqldict['Name'].append(name)
                    sqldict['City'].append(city)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['Cntry'].append('BO')
                    sqldict['RegCtry'].append('BO')
                    sqldict['RegCode'].append('ASFI')
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append("Regulated")
            except:
                pass

            

            sqldict = bourange_same_length_array(sqldict)
            


Working with BO ASFI 1
1
Nacional de Bolivia S.A.
La Paz
2
Mercantil Santa Cruz S.A.
La Paz
3
Bisa S.A.
La Paz
4
Crédito de Bolivia S.A.
La Paz
5
Económico S.A.
Santa Cruz
6
Ganadero S.A.
Santa Cruz
7
Solidario S.A.
La Paz
8
Fomento a Iniciativas Económicas S.A.
La Paz
9
De la Nación Argentina
Santa Cruz
10
Prodem S.A.
La Paz
11
Fortaleza S.A.
La Paz
1
De la Comunidad S.A.
Cochabamba
2
Ecofuturo S.A.
La Paz
1
La Primera
La Paz
2
La Promotora
Cochabamba
3
El Progreso
Oruro
1
Abierta “Jesús Nazareno” R.L.
Santa Cruz
2
Abierta “Fátima” R.L.
Santa Cruz
3
Abierta “San Martín de Porres” R.L.
Santa Cruz
4
Abierta “San Antonio” R.L.
Cochabamba
5
Abierta “Inca Huasi” R.L.
Cochabamba
6
Abierta “Quillacollo” R.L.
Quillacollo
7
Abierta “San José de Punata” R.L.
Cochabamba
8
Abierta “Loyola” R.L.
Cochabamba
9
Abierta “Pío X” R.L.
Cochabamba
10
Abierta “El Chorolque” R.L.
Tupiza
11
Abierta “San Pedro” R.L.
Cochabamba
12
Abierta “Catedral” R.L.
Potosí
13
Abierta “Asunción” R.L.

14
Abierta “Catedral 

In [17]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()

C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_24012\2242856004.py:7: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [18]:
df.to_csv('BO_ASFI_total_3.csv')